# Saudi/MENA-only PII augmentation dataset builder

This notebook creates a standalone English/Latin-script Saudi and MENA augmentation dataset.
It does not ingest any general-purpose PII corpus.

The builder prioritizes real public entity vocabularies, fictional identifiers that follow verified
Saudi formats, split-disjoint identities where practical, shared templates and numeric distributions
across all splits, strict span validation, deduplication, and detailed audit outputs.

Outputs:

- `saudi_mena_pii_dataset.csv`
- `saudi_mena_pii_summary.csv`
- `saudi_mena_pii_summary.xlsx`
- `saudi_mena_pii_manifest.json`

Full generation is the default. Set `PII_RUN_MODE=smoke` for a small offline-capable validation run.


In [ ]:
# Install only missing runtime packages in the current Jupyter kernel.
import importlib.util, os, subprocess, sys
from pathlib import Path

_start = Path.cwd().resolve()
REPO_ROOT = next(
    (root for root in (_start, *_start.parents)
     if (root / "model" / "data" / "saudi_PII_Builder.ipynb").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from inside the OnKith repository.")
BUILDER_DIR = REPO_ROOT / "model" / "data"
os.environ.setdefault("HF_HOME", os.getenv("PII_CACHE_DIR", str(BUILDER_DIR / ".pii_hf_cache")))
_required = {
    "datasets": "datasets>=4.0",
    "pandas": "pandas>=2.0",
    "openpyxl": "openpyxl>=3.1",
    "requests": "requests>=2.31",
    "tqdm": "tqdm>=4.66",
}
_missing = [spec for module, spec in _required.items() if importlib.util.find_spec(module) is None]
if _missing and os.getenv("PII_SKIP_INSTALL", "0") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
elif _missing:
    raise ImportError(f"Missing packages with PII_SKIP_INSTALL=1: {_missing}")


In [ ]:
import csv
import hashlib
import io
import json
import random
import re
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Any

import pandas as pd
import requests
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm

PRIVACY_DATA_DIR = REPO_ROOT / "model" / "privacy_data_v2"
if str(PRIVACY_DATA_DIR) not in sys.path:
    sys.path.insert(0, str(PRIVACY_DATA_DIR))
from ontology import ENTITIES as MODEL_V2_ENTITIES
from password_rules import classify_password_span

SEED = int(os.getenv("PII_SEED", "20260812"))
RUN_MODE = os.getenv("PII_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("PII_RUN_MODE must be 'full' or 'smoke'")

OUTPUT_DIR = Path(os.getenv("PII_OUTPUT_DIR", str(BUILDER_DIR))).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / "saudi_mena_pii_dataset.csv"
SUMMARY_CSV_PATH = OUTPUT_DIR / "saudi_mena_pii_summary.csv"
XLSX_PATH = OUTPUT_DIR / "saudi_mena_pii_summary.xlsx"
MANIFEST_PATH = OUTPUT_DIR / "saudi_mena_pii_manifest.json"
HF_TOKEN = os.getenv("HF_TOKEN") or None
NATIONAL_ADDRESS_API_KEY = os.getenv("NATIONAL_ADDRESS_API_KEY") or None
HF_DATASETS_CACHE = Path(os.environ["HF_HOME"]) / "datasets"
HF_DATASETS_CACHE.mkdir(parents=True, exist_ok=True)
REQUEST_TIMEOUT = int(os.getenv("PII_REQUEST_TIMEOUT", "90"))
RNG = random.Random(SEED)

UNIVERSAL_LABELS = tuple(MODEL_V2_ENTITIES)
DELETED_LABELS = {"USERNAME", "ALLERGIES", "HEIGHT", "WEIGHT"}
assert not DELETED_LABELS.intersection(UNIVERSAL_LABELS)

TARGETS = {label: 0 for label in UNIVERSAL_LABELS}
TARGETS.update({
    "GIVENNAME": 15_000, "SURNAME": 15_000, "STREET": 10_000, "CITY": 10_000,
    "ORGANISATION": 8_000, "IDCARDNUM": 8_000, "BANKNAME": 5_000, "ACCOUNTNUM": 5_000,
    "TELEPHONENUM": 5_000, "BUILDINGNUM": 5_000, "ZIPCODE": 5_000,
    "DRIVERLICENSENUM": 5_000, "PASSPORTNUM": 5_000, "TAXNUM": 5_000,
    "COUNTRY": 2_000, "CURRENCY": 3_000, "TIMEZONE": 1_000,
    "PASSWORD": 2_000,
})
SKIPPED_FORMATS = {
    "IDCARDNUM": "Skipped: no authoritative fictional-generation and checksum rule was established.",
    "DRIVERLICENSENUM": "Skipped: no authoritative Saudi licence-number format was established.",
    "PASSPORTNUM": "Skipped: no authoritative Saudi passport-number format was established.",
}
VERIFIED_GENERATED_LABELS = {
    "ACCOUNTNUM", "TELEPHONENUM", "BUILDINGNUM", "ZIPCODE", "TAXNUM", "PASSWORD"
}
REAL_POOL_LABELS = {
    "GIVENNAME", "SURNAME", "STREET", "CITY", "ORGANISATION", "BANKNAME",
    "COUNTRY", "CURRENCY", "TIMEZONE",
}

ROW_TARGET_MIN = int(os.getenv("PII_ROW_TARGET_MIN", "40000" if RUN_MODE == "full" else "120"))
ROW_TARGET_MAX = int(os.getenv("PII_ROW_TARGET_MAX", "60000" if RUN_MODE == "full" else "300"))
if not 0 < ROW_TARGET_MIN <= ROW_TARGET_MAX:
    raise ValueError("Require 0 < PII_ROW_TARGET_MIN <= PII_ROW_TARGET_MAX")
SMOKE_SCALE = float(os.getenv("PII_SMOKE_SCALE", "0.002"))


## 1. Real Saudi/MENA entity sources

Only Latin-script values enter the output. Arabic-script fields may be present in source datasets, but
they are never copied into generated text. Every accepted entity retains its source attribution.


In [ ]:
RESOURCE_AUDIT: list[dict[str, Any]] = []
REJECTED_EXAMPLES: defaultdict[str, list[str]] = defaultdict(list)

@dataclass(frozen=True)
class EntityRecord:
    value: str
    source: str

def audit_resource(resource: str, url: str, status: str, accepted: int, notes: str) -> None:
    RESOURCE_AUDIT.append({
        "resource": resource, "url": url, "status": status,
        "accepted_entities": int(accepted), "notes": notes,
    })

def stable_int(value: str, salt: str = "") -> int:
    return int.from_bytes(hashlib.blake2b(f"{salt}|{value}".encode("utf-8"), digest_size=8).digest(), "big")

def has_arabic(text: str) -> bool:
    return bool(re.search(r"[\u0600-\u06ff\u0750-\u077f\u08a0-\u08ff]", text))

def clean_latin_entity(value: Any, label: str, source: str, max_len: int = 120) -> str | None:
    raw = re.sub(r"\s+", " ", str(value or "")).strip(" \t\r\n,;|")
    normalized = unicodedata.normalize("NFKD", raw).encode("ascii", "ignore").decode("ascii")
    normalized = re.sub(r"\s+", " ", normalized).strip(" ,;|")
    valid = (
        bool(normalized) and len(normalized) <= max_len and not has_arabic(normalized)
        and bool(re.search(r"[A-Za-z]", normalized))
        and not bool(re.search(r"[^A-Za-z0-9 .&'()/-]", normalized))
    )
    if not valid:
        if len(REJECTED_EXAMPLES[label]) < 5:
            REJECTED_EXAMPLES[label].append(f"{source}: {raw[:80]!r}")
        return None
    return normalized

def dedup_records(records: list[EntityRecord]) -> list[EntityRecord]:
    chosen: dict[str, EntityRecord] = {}
    for record in records:
        key = re.sub(r"\s+", " ", record.value).strip().casefold()
        chosen.setdefault(key, record)
    return sorted(chosen.values(), key=lambda item: item.value.casefold())

GIVEN_FALLBACK = [
    "Mohammed", "Ahmed", "Abdulaziz", "Abdullah", "Faisal", "Khalid", "Salman", "Saad",
    "Omar", "Yousef", "Ibrahim", "Hassan", "Hussein", "Tariq", "Ziyad", "Rayan",
    "Amina", "Noura", "Maha", "Lama", "Sara", "Hala", "Reem", "Dana", "Lina", "Mariam",
    "Fatimah", "Aisha", "Jawaher", "Manal", "Rana", "Razan", "Lujain", "Nada", "Basma", "Wafa",
]

def load_given_names() -> list[EntityRecord]:
    source = "Muslim Names Dataset"
    url = "https://huggingface.co/datasets/wpacademy/muslim-names-dataset"
    fallback = [EntityRecord(name, "curated Saudi/MENA smoke fallback") for name in GIVEN_FALLBACK]
    if RUN_MODE == "smoke":
        audit_resource(source, url, "smoke fallback", len(fallback), "Offline smoke vocabulary")
        return fallback
    records: list[EntityRecord] = []
    try:
        dataset = load_dataset(
            "wpacademy/muslim-names-dataset", split="train", streaming=True,
            token=HF_TOKEN, cache_dir=str(HF_DATASETS_CACHE),
        )
        for row in dataset:
            value = clean_latin_entity(row.get("english_name"), "GIVENNAME", source, 40)
            if value and not re.search(r"\b(?:bin|bint|ibn)\b", value, re.I):
                records.append(EntityRecord(value, source))
        records = dedup_records(records)
        audit_resource(source, url, "loaded", len(records), "CC0 real Muslim given-name vocabulary")
    except Exception as exc:
        audit_resource(source, url, "fallback", len(fallback), repr(exc))
        records = fallback

    pairs_url = "https://huggingface.co/datasets/Sayedyousef/arabic-name-pairs/resolve/main/dict_FINAL.json"
    try:
        response = requests.get(pairs_url, timeout=REQUEST_TIMEOUT, headers={"User-Agent": "Saudi-MENA-PII-builder/1.0"})
        response.raise_for_status()
        pairs = response.json()
        cross_checked = sum(name.value.casefold() in pairs for name in records)
        audit_resource("Arabic Name Pairs", pairs_url, "cross-check only", cross_checked,
                       "CC-BY-4.0 name-token lexicon; not used to infer surname/given-name class")
    except Exception as exc:
        audit_resource("Arabic Name Pairs", pairs_url, "unavailable", 0, repr(exc))
    return records or fallback

SAUDI_SURNAMES = [
    "Al Harbi", "Al Qahtani", "Al Otaibi", "Al Mutairi", "Al Dosari", "Al Shammari",
    "Al Ghamdi", "Al Zahrani", "Al Shehri", "Al Anazi", "Al Subaie", "Al Malki",
    "Al Amri", "Al Juhani", "Al Balawi", "Al Rashidi", "Al Sulami", "Al Hazmi",
    "Al Qarni", "Al Asiri", "Al Harthi", "Al Thaqafi", "Al Fifi", "Al Yamani",
    "Al Tamimi", "Al Khaldi", "Al Nasser", "Al Saleh", "Al Saud", "Al Faisal",
    "Al Ibrahim", "Al Hamad", "Al Majed", "Al Rashed", "Al Sayegh", "Al Turki",
    "Al Khatib", "Al Faris", "Al Mansour", "Al Hakim", "Al Najjar", "Al Bahri",
    "Al Zamil", "Al Jomaih", "Al Rajhi", "Al Olayan", "Al Hokair", "Al Muhaidib",
    "Al Fozan", "Alissa", "Al Gosaibi", "Al Khereiji", "Al Suwaidi", "Al Dabbagh",
    "Al Bassam", "Al Dughaither", "Al Kholifey", "Al Sadhan", "Al Subeaei", "Al Rumaih",
    "Al Twaijri", "Al Musallam", "Al Habib", "Al Obaid", "Al Eisa", "Al Jasser",
    "Al Rasheed", "Al Kadi", "Al Moammar", "Al Babtain", "Al Akeel", "Al Sudairi",
    "Al Shaikh", "Al Angari", "Al Dakhil", "Al Zaid", "Al Bawardi", "Al Jarallah",
]

def load_surnames() -> list[EntityRecord]:
    source = "curated public Saudi family and tribal names"
    records = dedup_records([EntityRecord(name, source) for name in SAUDI_SURNAMES])
    audit_resource("Saudi family names", "curated from public Saudi institutional and biographical references",
                   "conservative curated pool", len(records),
                   "No fictional combinations; use is capped to avoid manufacturing diversity")
    audit_resource("ANETAC", "https://arxiv.org/abs/1907.03110", "not used for surnames", 0,
                   "Person entities are full names; the published class does not identify surname boundaries reliably")
    return records

STREET_FALLBACK = [
    "King Abdulaziz Road", "King Fahd Road", "King Abdullah Road", "Prince Turki Road",
    "Olaya Street", "Al Washm Street", "Makkah Road", "Takhassusi Street",
    "Eastern Ring Road", "Northern Ring Road", "Prince Sultan Road", "Palestine Street",
    "Al Madinah Road", "King Saud Road", "Imam Saud Road", "Tahlia Street",
]

def load_rcrc_streets() -> list[EntityRecord]:
    source = "RCRC Traffic Intersections"
    url = ("https://opendata.rcrc.gov.sa/api/explore/v2.1/catalog/datasets/"
           "traffic-intersections-by-main-street-and-cross-street-2024/exports/csv")
    fallback = [EntityRecord(value, source + " curated fallback") for value in STREET_FALLBACK]
    if RUN_MODE == "smoke":
        audit_resource(source, url, "smoke fallback", len(fallback), "Offline smoke vocabulary")
        return fallback
    try:
        response = requests.get(url, params={"lang": "en", "timezone": "Asia/Riyadh", "use_labels": "true"},
                                timeout=REQUEST_TIMEOUT, headers={"User-Agent": "Saudi-MENA-PII-builder/1.0"})
        response.raise_for_status()
        frame = pd.read_csv(io.BytesIO(response.content))
        columns = [column for column in frame.columns if "street" in str(column).casefold()]
        records = []
        for column in columns:
            for raw in frame[column].dropna().astype(str):
                value = clean_latin_entity(raw, "STREET", source, 100)
                if value:
                    records.append(EntityRecord(value, source))
        records = dedup_records(records + fallback)
        audit_resource(source, url, "loaded", len(records), "Official public English intersection export")
        return records
    except Exception as exc:
        audit_resource(source, url, "fallback", len(fallback), repr(exc))
        return fallback

CITY_FALLBACK = [
    "Riyadh", "Jeddah", "Makkah", "Medina", "Dammam", "Khobar", "Tabuk", "Abha", "Taif",
    "Yanbu", "Jazan", "Hail", "Buraidah", "Najran", "Al Hofuf", "Jubail", "Al Kharj",
    "Dhahran", "Al Bahah", "Sakaka", "Arar", "Qatif", "Unaizah", "Ras Tanura", "Khamis Mushait",
    "Hafar Al Batin", "AlUla", "Rabigh", "Diriyah", "Dawadmi", "Sharurah", "Al Majmaah",
]

def wikidata_saudi_records(kind: str) -> list[EntityRecord]:
    class_qid = "Q515" if kind == "city" else "Q43229"
    limit = 3000 if kind == "city" else 5000
    query = f"""SELECT DISTINCT ?item ?label WHERE {{
      ?item wdt:P17 wd:Q851; wdt:P31/wdt:P279* wd:{class_qid}.
      ?item rdfs:label ?label FILTER(LANG(?label)='en')
    }} LIMIT {limit}"""
    url = "https://query.wikidata.org/sparql"
    if RUN_MODE == "smoke":
        return []
    try:
        response = requests.get(url, params={"query": query, "format": "json"}, timeout=REQUEST_TIMEOUT,
                                headers={"User-Agent": "Saudi-MENA-PII-builder/1.0"})
        response.raise_for_status()
        records = []
        for binding in response.json().get("results", {}).get("bindings", []):
            value = clean_latin_entity(binding.get("label", {}).get("value"),
                                        "CITY" if kind == "city" else "ORGANISATION", "Wikidata", 120)
            if value:
                records.append(EntityRecord(value, f"Wikidata Saudi {kind} labels"))
        records = dedup_records(records)
        audit_resource(f"Wikidata Saudi {kind} labels", url, "loaded", len(records),
                       "CC0 English labels filtered to country=Saudi Arabia")
        return records
    except Exception as exc:
        audit_resource(f"Wikidata Saudi {kind} labels", url, "unavailable", 0, repr(exc))
        return []

def load_cities() -> list[EntityRecord]:
    source = "National Address API - KSA Cities"
    records = [EntityRecord(value, source + " curated fallback") for value in CITY_FALLBACK]
    url = "https://apina.address.gov.sa/NationalAddress/v3.1/lookup/cities"
    if NATIONAL_ADDRESS_API_KEY and RUN_MODE == "full":
        try:
            response = requests.get(url, params={"format": "JSON", "regionid": -1, "language": "E",
                                                "api_key": NATIONAL_ADDRESS_API_KEY}, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            for item in response.json().get("Cities") or []:
                value = clean_latin_entity(item.get("Name"), "CITY", source, 80)
                if value:
                    records.append(EntityRecord(value, source))
            audit_resource(source, "https://api.address.gov.sa/cities", "loaded", len(records), "Official API")
        except Exception as exc:
            audit_resource(source, "https://api.address.gov.sa/cities", "fallback", len(records), repr(exc))
    else:
        audit_resource(source, "https://api.address.gov.sa/cities", "documented fallback", len(records),
                       "Set NATIONAL_ADDRESS_API_KEY to enable the official lookup")
    return dedup_records(records + wikidata_saudi_records("city"))

OFFICIAL_ORGANISATIONS = [
    "Ministry of Interior", "Ministry of Foreign Affairs", "Ministry of Justice", "Ministry of Education",
    "Ministry of Health", "Ministry of Finance", "Ministry of Commerce", "Ministry of Investment",
    "Ministry of Human Resources and Social Development", "Ministry of Transport and Logistic Services",
    "Saudi Data and AI Authority", "General Authority for Statistics", "Zakat Tax and Customs Authority",
    "Communications Space and Technology Commission", "Saudi Central Bank", "Capital Market Authority",
    "Saudi Post", "Royal Commission for Riyadh City", "Saudi Food and Drug Authority",
    "Saudi Standards Metrology and Quality Organization", "King Saud University", "King Abdulaziz University",
    "King Fahd University of Petroleum and Minerals", "King Abdullah University of Science and Technology",
    "Princess Nourah bint Abdulrahman University", "Imam Mohammad Ibn Saud Islamic University",
    "Umm Al-Qura University", "Islamic University of Madinah", "King Faisal University", "King Khalid University",
    "Qassim University", "Taibah University", "Taif University", "University of Jeddah", "Jazan University",
    "University of Tabuk", "Najran University", "University of Hail", "Al Jouf University", "Bisha University",
]

def load_organisations() -> list[EntityRecord]:
    source = "Saudi official government and university nomenclature"
    url = "https://www.boe.gov.sa/en/Translation/Pages/GovernmentLabels.aspx"
    records = [EntityRecord(value, source) for value in OFFICIAL_ORGANISATIONS]
    records.extend(wikidata_saudi_records("organisation"))
    records = dedup_records(records)
    audit_resource(source, url, "curated official English names", len(records),
                   "Government bodies and Saudi universities; no invented organisations")
    return records

SAMA_BANKS = [
    "Saudi National Bank", "Al Rajhi Bank", "Riyad Bank", "Saudi Awwal Bank", "Arab National Bank",
    "Alinma Bank", "Banque Saudi Fransi", "Saudi Investment Bank", "Bank AlJazira", "Bank Albilad",
    "Gulf International Bank Saudi Arabia", "D360 Bank",
]

def load_banks() -> list[EntityRecord]:
    source = "Saudi Central Bank Licensed Banks"
    url = "https://www.sama.gov.sa/en-us/supervision/licenseentities/pages/licensedbanks.aspx"
    records = [EntityRecord(value, source) for value in SAMA_BANKS]
    audit_resource(source, url, "curated from official list", len(records), "Closed finite vocabulary")
    return records


## 2. Leakage-aware entity pools and fictional Saudi-format values

Open vocabularies are partitioned by identity before generation. Banks, country, currency, and timezone
are audited closed-vocabulary exceptions. Numeric generators ignore the split and use one global sequence.


In [ ]:
CLOSED_LABELS = {"BANKNAME", "COUNTRY", "CURRENCY", "TIMEZONE"}
SPLIT_EXCEPTIONS = [
    {"label": label, "reason": "closed finite vocabulary", "mitigation": "distinct generated sentences"}
    for label in sorted(CLOSED_LABELS)
]

def partition_records(records: list[EntityRecord], label: str) -> dict[str, list[EntityRecord]]:
    records = dedup_records(records)
    if label in CLOSED_LABELS:
        return {split: list(records) for split in ("train", "validation", "test")}
    order = list(records)
    random.Random(stable_int(label, "entity-partition")).shuffle(order)
    count = len(order)
    train_end = max(1, int(count * 0.8)) if count else 0
    validation_end = max(train_end + 1, int(count * 0.9)) if count >= 3 else train_end
    validation_end = min(validation_end, max(train_end, count - 1))
    return {
        "train": order[:train_end],
        "validation": order[train_end:validation_end],
        "test": order[validation_end:],
    }

def build_entity_pools() -> dict[str, dict[str, list[EntityRecord]]]:
    raw = {
        "GIVENNAME": load_given_names(),
        "SURNAME": load_surnames(),
        "STREET": load_rcrc_streets(),
        "CITY": load_cities(),
        "ORGANISATION": load_organisations(),
        "BANKNAME": load_banks(),
        "COUNTRY": [EntityRecord("Saudi Arabia", "official country name")],
        "CURRENCY": [EntityRecord("Saudi riyal", "Saudi currency vocabulary"),
                     EntityRecord("SAR", "ISO 4217 currency code")],
        "TIMEZONE": [EntityRecord("Asia/Riyadh", "IANA time zone database"),
                     EntityRecord("UTC+03:00", "Saudi standard time")],
    }
    audit_resource("LiStreet Saudi addresses", "https://listreet.com/pages/saudi-arabia", "not fetched", 0,
                   "Machine-readable access and reuse licence were not established")
    pools = {label: partition_records(records, label) for label, records in raw.items()}
    for label, split_pools in pools.items():
        if label in CLOSED_LABELS:
            continue
        if any(not split_pools[split] for split in ("train", "validation", "test")):
            raise ValueError(f"Entity pool too small for disjoint split partition: {label}")
    return pools

class PoolCycler:
    def __init__(self, pools: dict[str, dict[str, list[EntityRecord]]]) -> None:
        self.pools = pools
        self.orders: dict[tuple[str, str], list[EntityRecord]] = {}
        self.indices = Counter()

    def next(self, label: str, split: str) -> EntityRecord:
        key = (label, split)
        if key not in self.orders:
            values = list(self.pools[label][split])
            random.Random(stable_int("|".join(key), "pool-order")).shuffle(values)
            self.orders[key] = values
        values = self.orders[key]
        if not values:
            raise KeyError(f"No entities for {label}/{split}")
        index = self.indices[key]
        self.indices[key] += 1
        return values[index % len(values)]

class ValueFactory:
    def __init__(self, pools: dict[str, dict[str, list[EntityRecord]]]) -> None:
        self.pool = PoolCycler(pools)
        self.counters = Counter()
        self.used_values: defaultdict[str, set[str]] = defaultdict(set)

    def _number(self, label: str) -> int:
        self.counters[label] += 1
        return stable_int(f"{label}|{self.counters[label]}", "generated-value")

    def _unique(self, label: str, formatter) -> str:
        while True:
            value = formatter(self._number(label))
            if value not in self.used_values[label]:
                self.used_values[label].add(value)
                return value

    @staticmethod
    def _iban(bban: str) -> str:
        check = 98 - (int(bban + "281000") % 97)
        return f"SA{check:02d}{bban}"

    def get(self, label: str, split: str) -> EntityRecord:
        if label in self.pool.pools:
            return self.pool.next(label, split)
        if label == "ACCOUNTNUM":
            value = self._unique(label, lambda n: self._iban(f"{n % 100:02d}{(n * 982451653) % 10**18:018d}"))
            return EntityRecord(value, "fictional Saudi IBAN; SAMA format and ISO 13616 check digits")
        if label == "TELEPHONENUM":
            mobile_prefixes = (50, 53, 54, 55, 56, 57, 58, 59)
            def phone(n: int) -> str:
                prefix = mobile_prefixes[n % len(mobile_prefixes)]
                subscriber = (n * 7919) % 10_000_000
                return f"+966 {prefix} {subscriber // 10_000:03d} {subscriber % 10_000:04d}"
            return EntityRecord(self._unique(label, phone), "fictional number; Saudi CST mobile format")
        if label == "BUILDINGNUM":
            # A four-digit building number is not a person identity and may repeat across unrelated addresses.
            value = f"{1000 + (self._number(label) % 9000):04d}"
            return EntityRecord(value, "fictional four-digit Saudi National Address building number")
        if label == "ZIPCODE":
            value = self._unique(label, lambda n: f"{10000 + (n % 90000):05d}")
            return EntityRecord(value, "fictional five-digit Saudi National Address postal code")
        if label == "TAXNUM":
            value = self._unique(label, lambda n: f"3{n % 10**13:013d}3")
            return EntityRecord(value, "fictional 15-digit Saudi VAT number; ZATCA format")
        if label == "PASSWORD":
            words = ("Palm", "Falcon", "Dune", "Cedar", "Oasis", "Pearl", "Tuwaiq", "Najd")
            value = self._unique(label, lambda n: f"{words[n % len(words)]}!{n % 100000:05d}R")
            return EntityRecord(value, "fictional local generator")
        raise KeyError(f"No reliable Saudi/MENA value generator for {label}")


In [ ]:
# Static format assertions document the supported fictional identifier distributions.
def validate_generated_format(label: str, value: str) -> bool:
    patterns = {
        "ACCOUNTNUM": r"SA\d{22}",
        "TELEPHONENUM": r"\+966 (?:50|53|54|55|56|57|58|59) \d{3} \d{4}",
        "BUILDINGNUM": r"\d{4}",
        "ZIPCODE": r"\d{5}",
        "TAXNUM": r"3\d{13}3",
        "PASSWORD": r"[A-Za-z]+![0-9]{5}R",
    }
    return bool(re.fullmatch(patterns[label], value))

def validate_iban(value: str) -> bool:
    rearranged = value[4:] + value[:4]
    numeric = "".join(str(ord(char) - 55) if char.isalpha() else char for char in rearranged)
    return len(value) == 24 and value.startswith("SA") and int(numeric) % 97 == 1


## 3. Shared English templates

Train, validation, and test use this exact template and phrase pool. Split selection affects only which
partitioned entities are available; it never changes wording, prefixes, numeric ranges, or formats.


In [ ]:
GENERAL_CONTEXT = {
    "documents": ["service form", "customer letter", "delivery note", "account request", "supplier record",
                  "permit form", "contact sheet", "payment instruction", "case note", "booking record"],
    "purposes": ["identity confirmation", "address confirmation", "payment processing", "licence renewal",
                 "shipment delivery", "customer support", "supplier onboarding", "account maintenance"],
    "departments": ["customer care", "operations", "finance", "logistics", "administration", "compliance"],
    "channels": ["secure portal", "service desk", "branch office", "support line", "customer application"],
}

MULTI_TEMPLATES: list[list[str]] = [
    ["For {purpose}, contact ", "GIVENNAME", " at ", "TELEPHONENUM", " in ", "CITY", "."],
    ["The {document} lists ", "GIVENNAME", " ", "SURNAME", " at building ", "BUILDINGNUM", ", ", "STREET", ", ", "CITY", " ", "ZIPCODE", "."],
    ["Send the parcel to building ", "BUILDINGNUM", " on ", "STREET", " in ", "CITY", " ", "ZIPCODE", ", ", "COUNTRY", "."],
    ["The {department} team asked ", "GIVENNAME", " to confirm account ", "ACCOUNTNUM", " with ", "BANKNAME", "."],
    ["A payment for ", "GIVENNAME", " will be sent through ", "BANKNAME", " to ", "ACCOUNTNUM", " in ", "CURRENCY", "."],
    ["The supplier ", "ORGANISATION", " submitted VAT number ", "TAXNUM", " for settlement in ", "CURRENCY", "."],
    ["The {document} from ", "ORGANISATION", " gives account ", "ACCOUNTNUM", " at ", "BANKNAME", "."],
    ["Call ", "TELEPHONENUM", " before visiting ", "ORGANISATION", " in ", "CITY", "."],
    ["The appointment with ", "ORGANISATION", " in ", "CITY", " uses timezone ", "TIMEZONE", "."],
    ["The recipient ", "GIVENNAME", " ", "SURNAME", " can be reached at ", "TELEPHONENUM", "."],
    ["Record ", "GIVENNAME", " for the delivery to ", "BUILDINGNUM", " ", "STREET", ", ", "CITY", "."],
    ["The {department} office in ", "CITY", " received a request from ", "ORGANISATION", "."],
    ["Use ", "PASSWORD", " as the temporary password for ", "GIVENNAME", " on the {channel}."],
    ["The customer named ", "GIVENNAME", " provided ", "ACCOUNTNUM", " and ", "TELEPHONENUM", "."],
    ["Deliver the {document} to ", "ORGANISATION", " at ", "BUILDINGNUM", " ", "STREET", ", ", "CITY", " ", "ZIPCODE", "."],
    ["For {purpose}, ", "BANKNAME", " recorded VAT number ", "TAXNUM", " and account ", "ACCOUNTNUM", "."],
    ["The address in ", "COUNTRY", " is building ", "BUILDINGNUM", ", ", "STREET", ", ", "CITY", " ", "ZIPCODE", "."],
    ["The {document} names ", "GIVENNAME", " and cites ", "ORGANISATION", " in ", "CITY", "."],
]

LABEL_TERMS = {
    "GIVENNAME": "given name", "SURNAME": "family name", "STREET": "street", "CITY": "city",
    "ORGANISATION": "organisation", "BANKNAME": "bank", "ACCOUNTNUM": "Saudi IBAN",
    "TELEPHONENUM": "contact number", "BUILDINGNUM": "building number", "ZIPCODE": "postal code",
    "TAXNUM": "VAT registration number", "COUNTRY": "country", "CURRENCY": "currency",
    "TIMEZONE": "timezone", "PASSWORD": "temporary password",
}

SINGLE_TEMPLATES = [
    "The {document} records {value} as the {term} for {purpose}.",
    "Please enter {value} in the {term} field on the {channel}.",
    "The {department} team received {value} for the {term}.",
    "For {purpose}, the supplied {term} is {value}.",
    "The customer confirmed {value} as the correct {term}.",
    "A {document} submitted through the {channel} contains {value} as its {term}.",
]

def context_words(rng: random.Random) -> dict[str, str]:
    return {key[:-1] if key.endswith("s") else key: rng.choice(values) for key, values in GENERAL_CONTEXT.items()}

def template_labels(template: list[str]) -> list[str]:
    return [part for part in template if part in UNIVERSAL_LABELS]

def render_parts(parts: list[Any]) -> tuple[str, list[dict], list[str], list[str]]:
    text_parts: list[str] = []
    spans: list[dict] = []
    sources: list[str] = []
    span_sources: list[str] = []
    cursor = 0
    for part in parts:
        if isinstance(part, tuple):
            label, record = part
            start = cursor
            text_parts.append(record.value)
            cursor += len(record.value)
            spans.append({"start": start, "end": cursor, "label": label, "value": record.value})
            sources.append(record.source)
            span_sources.append(record.source)
        else:
            text_parts.append(part)
            cursor += len(part)
    return "".join(text_parts), spans, sorted(set(sources)), span_sources

class TemplateEngine:
    def __init__(self, factory: ValueFactory, seed: int) -> None:
        self.factory = factory
        self.rng = random.Random(seed)
        self.counter = 0

    def make_single(self, label: str, split: str) -> tuple[str, list[dict], list[str], list[str]]:
        record = self.factory.get(label, split)
        context = context_words(self.rng)
        if label == "PASSWORD":
            template = self.rng.choice([
                "The temporary password is {value}.",
                "Use {value} to log in to the {channel}.",
                "I changed my account password to {value}.",
            ])
        else:
            template = self.rng.choice(SINGLE_TEMPLATES)
        marker = "__ENTITY_VALUE__"
        rendered = template.format(value=marker, term=LABEL_TERMS[label], **context)
        before, after = rendered.split(marker)
        return render_parts([before, (label, record), after])

    def make_multi(self, focus: str, split: str, counts: Counter, targets: dict[str, int]) -> tuple[str, list[dict], list[str], list[str]]:
        candidates = [template for template in MULTI_TEMPLATES if focus in template
                      and not ("SURNAME" in template and counts["SURNAME"] >= targets["SURNAME"]
                               and focus != "SURNAME")]
        if not candidates:
            return self.make_single(focus, split)
        scored = []
        for template in candidates:
            labels = template_labels(template)
            score = sum(max(0, targets.get(label, 0) - counts[label]) for label in labels)
            scored.append((score, self.rng.random(), template))
        template = max(scored, key=lambda item: (item[0], item[1]))[2]
        context = context_words(self.rng)
        parts: list[Any] = []
        for token in template:
            if token in UNIVERSAL_LABELS:
                parts.append((token, self.factory.get(token, split)))
            else:
                parts.append(token.format(**context))
        return render_parts(parts)

    def make_row(self, focus: str, split: str, counts: Counter, targets: dict[str, int], multi: bool) -> dict:
        self.counter += 1
        text, spans, sources, span_sources = (self.make_multi(focus, split, counts, targets)
                                if multi else self.make_single(focus, split))
        identity = hashlib.blake2b(f"{self.counter}|{text}".encode("utf-8"), digest_size=10).hexdigest()
        return {
            "id": f"saudi-mena:{identity}", "text": text, "spans": spans,
            "labels_present": sorted({span["label"] for span in spans}),
            "source_dataset": "Saudi/MENA augmentation", "source_split": "generated",
            "final_split": split, "is_saudi": True,
            "generation_type": "synthetic" if multi else "replacement",
            "entity_source": " | ".join(sources), "_span_sources": span_sources,
        }


## 4. Span validation, deduplication, and audits


In [ ]:
CSV_COLUMNS = [
    "id", "text", "spans", "labels_present", "source_dataset", "source_split",
    "final_split", "is_saudi", "generation_type", "entity_source",
]

@dataclass
class QualityAudit:
    candidates: int = 0
    rows_written: int = 0
    duplicate_rows_removed: int = 0
    invalid_rows_removed: int = 0
    invalid_spans_removed: int = 0
    non_latin_rows_removed: int = 0

class Deduplicator:
    def __init__(self) -> None:
        self.fingerprints: set[str] = set()

    @staticmethod
    def normalized(text: str) -> str:
        return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text)).strip().casefold()

    def accept(self, row: dict) -> bool:
        payload = self.normalized(row["text"])
        fingerprint = hashlib.blake2b(payload.encode("utf-8"), digest_size=16).hexdigest()
        if fingerprint in self.fingerprints:
            return False
        self.fingerprints.add(fingerprint)
        return True

@dataclass
class Stats:
    rows_by_split: Counter = field(default_factory=Counter)
    spans_by_split: Counter = field(default_factory=Counter)
    rows_by_generation: Counter = field(default_factory=Counter)
    spans_by_generation: Counter = field(default_factory=Counter)
    label_rows: Counter = field(default_factory=Counter)
    label_spans: Counter = field(default_factory=Counter)
    label_split_spans: Counter = field(default_factory=Counter)
    label_generation_spans: Counter = field(default_factory=Counter)
    label_entities: defaultdict = field(default_factory=lambda: defaultdict(set))
    label_sources: defaultdict = field(default_factory=lambda: defaultdict(set))
    source_rows: Counter = field(default_factory=Counter)
    source_spans: Counter = field(default_factory=Counter)

    def add(self, row: dict) -> None:
        split = row["final_split"]
        generation = row["generation_type"]
        labels = {span["label"] for span in row["spans"]}
        self.rows_by_split[split] += 1
        self.spans_by_split[split] += len(row["spans"])
        self.rows_by_generation[generation] += 1
        self.spans_by_generation[generation] += len(row["spans"])
        sources = [source.strip() for source in row["entity_source"].split("|") if source.strip()]
        for source in sources:
            self.source_rows[source] += 1
        for label in labels:
            self.label_rows[label] += 1
        for span, source in zip(row["spans"], row["_span_sources"]):
            label = span["label"]
            self.label_spans[label] += 1
            self.label_split_spans[(label, split)] += 1
            self.label_generation_spans[(label, generation)] += 1
            self.label_entities[label].add(span["value"].casefold())
            self.label_sources[label].add(source)
            self.source_spans[source] += 1

def clean_and_validate_row(row: dict, quality: QualityAudit) -> dict | None:
    quality.candidates += 1
    text = row.get("text", "")
    if not isinstance(text, str) or not text.strip():
        quality.invalid_rows_removed += 1
        return None
    if has_arabic(text):
        quality.non_latin_rows_removed += 1
        return None
    valid_spans = []
    valid_span_sources = []
    original_sources = list(row.get("_span_sources", []))
    previous_end = -1
    indexed_spans = list(enumerate(row.get("spans", [])))
    for index, span in sorted(indexed_spans, key=lambda item: (item[1].get("start", -1), item[1].get("end", -1))):
        start, end = span.get("start"), span.get("end")
        valid = (
            isinstance(start, int) and isinstance(end, int) and 0 <= start < end <= len(text)
            and span.get("label") in UNIVERSAL_LABELS
            and text[start:end] == span.get("value")
            and start >= previous_end
        )
        if not valid:
            quality.invalid_spans_removed += 1
            continue
        valid_spans.append({key: span[key] for key in ("start", "end", "label", "value")})
        valid_span_sources.append(original_sources[index] if index < len(original_sources) else "unknown")
        previous_end = end
    if not valid_spans:
        quality.invalid_rows_removed += 1
        return None
    row["spans"] = valid_spans
    row["_span_sources"] = valid_span_sources
    row["labels_present"] = sorted({span["label"] for span in valid_spans})
    assert row["final_split"] in {"train", "validation", "test"}
    assert row["generation_type"] in {"synthetic", "replacement"}
    assert row["is_saudi"] is True
    return row

def serialized_row(row: dict) -> dict:
    output = {key: row[key] for key in CSV_COLUMNS}
    output["spans"] = json.dumps(row["spans"], ensure_ascii=False, separators=(",", ":"))
    output["labels_present"] = json.dumps(row["labels_present"], ensure_ascii=False, separators=(",", ":"))
    return output


In [ ]:
def exact_split_schedule(limit: int, seed: int) -> list[str]:
    schedule = []
    rng = random.Random(seed)
    while len(schedule) < limit:
        block = ["train"] * 80 + ["validation"] * 10 + ["test"] * 10
        rng.shuffle(block)
        schedule.extend(block)
    return schedule[:limit]

def effective_targets(pools: dict[str, dict[str, list[EntityRecord]]]) -> dict[str, int]:
    scale = SMOKE_SCALE if RUN_MODE == "smoke" else 1.0
    targets = {label: (max(1, round(target * scale)) if target else 0) for label, target in TARGETS.items()}
    for label in SKIPPED_FORMATS:
        targets[label] = 0
    # Do not repeat a tiny surname list merely to hit a quota.
    unique_surnames = len({record.value.casefold() for split in pools["SURNAME"].values() for record in split})
    targets["SURNAME"] = min(targets["SURNAME"], unique_surnames)
    return targets

def choose_focus(counts: Counter, targets: dict[str, int], rng: random.Random) -> str:
    unmet = [label for label, target in targets.items() if target > counts[label]]
    if unmet:
        scored = [((targets[label] - counts[label]) / max(1, targets[label]) + rng.random() * 0.03, label)
                  for label in unmet]
        return max(scored)[1]
    fallback = [label for label in targets if (label in REAL_POOL_LABELS or label in VERIFIED_GENERATED_LABELS)
                and label != "SURNAME"]
    return rng.choice(fallback)

def build_dataset() -> tuple[Stats, QualityAudit, dict[str, int], dict[str, dict[str, list[EntityRecord]]]]:
    pools = build_entity_pools()
    targets = effective_targets(pools)
    factory = ValueFactory(pools)
    engine = TemplateEngine(factory, SEED)
    stats = Stats()
    quality = QualityAudit()
    dedup = Deduplicator()
    split_schedule = exact_split_schedule(ROW_TARGET_MAX, SEED)
    rng = random.Random(SEED + 1)

    with CSV_PATH.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        attempts = 0
        progress = tqdm(total=ROW_TARGET_MIN, desc="Saudi/MENA rows", unit="rows")
        while quality.rows_written < ROW_TARGET_MAX:
            all_reached = all(stats.label_spans[label] >= target for label, target in targets.items())
            if quality.rows_written >= ROW_TARGET_MIN and all_reached:
                break
            attempts += 1
            if attempts > ROW_TARGET_MAX * 20:
                raise RuntimeError("Generation could not produce enough unique valid rows")
            split = split_schedule[quality.rows_written]
            focus = choose_focus(stats.label_spans, targets, rng)
            multi = rng.random() < 0.55
            row = engine.make_row(focus, split, stats.label_spans, targets, multi)
            row = clean_and_validate_row(row, quality)
            if row is None:
                continue
            if not dedup.accept(row):
                quality.duplicate_rows_removed += 1
                continue
            for span in row["spans"]:
                if span["label"] in VERIFIED_GENERATED_LABELS:
                    assert validate_generated_format(span["label"], span["value"])
                    if span["label"] == "ACCOUNTNUM":
                        assert validate_iban(span["value"])
            writer.writerow(serialized_row(row))
            stats.add(row)
            quality.rows_written += 1
            if quality.rows_written <= ROW_TARGET_MIN:
                progress.update(1)
        progress.close()
    return stats, quality, targets, pools


## 5. Detailed summaries and exports


In [ ]:
def label_notes(label: str, stats: Stats, pools: dict) -> str:
    if label in SKIPPED_FORMATS:
        return SKIPPED_FORMATS[label]
    if label == "SURNAME":
        return "Real curated surnames only; fictional composition is prohibited, so the 15k target is not forced."
    if label == "GIVENNAME":
        return "Real Muslim given names; unique_entity_count should be audited before downstream combination."
    if label in VERIFIED_GENERATED_LABELS:
        return "Values are fictional and generated from one shared split-independent Saudi format."
    if TARGETS[label] == 0 and stats.label_spans[label] == 0:
        return "Kept in the universal taxonomy but not targeted by this Saudi/MENA augmenter."
    if label in CLOSED_LABELS:
        return "Closed vocabulary may appear across splits; complete sentence text remains deduplicated."
    return "Real public entity pool; identity partitions are disjoint across splits."

def per_label_summary(stats: Stats, pools: dict) -> pd.DataFrame:
    rows = []
    for label in UNIVERSAL_LABELS:
        rejected = SKIPPED_FORMATS.get(label, "") or " | ".join(REJECTED_EXAMPLES.get(label, []))
        rows.append({
            "label": label,
            "total_rows": stats.label_rows[label],
            "total_spans": stats.label_spans[label],
            "train_spans": stats.label_split_spans[(label, "train")],
            "validation_spans": stats.label_split_spans[(label, "validation")],
            "test_spans": stats.label_split_spans[(label, "test")],
            "unique_entity_count": len(stats.label_entities[label]),
            "source_datasets_used": " | ".join(sorted(stats.label_sources[label])),
            "synthetic_spans": stats.label_generation_spans[(label, "synthetic")],
            "replacement_spans": stats.label_generation_spans[(label, "replacement")],
            "target_spans": TARGETS[label],
            "target_reached": stats.label_spans[label] >= TARGETS[label] if TARGETS[label] else None,
            "skipped_rejected_examples": rejected,
            "notes": label_notes(label, stats, pools),
        })
    return pd.DataFrame(rows)

def write_summaries(stats: Stats, quality: QualityAudit, pools: dict) -> dict[str, pd.DataFrame]:
    label_summary = per_label_summary(stats, pools)
    label_summary.to_csv(SUMMARY_CSV_PATH, index=False, encoding="utf-8")
    target_gaps = label_summary[(label_summary["target_spans"] > 0) &
                                (label_summary["total_spans"] < label_summary["target_spans"])].copy()
    skipped_labels = pd.DataFrame([
        {"label": label, "target_spans": TARGETS[label], "reason": reason}
        for label, reason in SKIPPED_FORMATS.items()
    ] + [
        {"label": label, "target_spans": 0,
         "reason": "Kept in universal taxonomy; no reliable Saudi-specific augmentation target requested."}
        for label in UNIVERSAL_LABELS if TARGETS[label] == 0 and label not in SKIPPED_FORMATS
    ])
    sheets = {
        "label_summary": label_summary,
        "split_summary": pd.DataFrame([
            {"final_split": split, "rows": stats.rows_by_split[split], "spans": stats.spans_by_split[split]}
            for split in ("train", "validation", "test")
        ]),
        "generation_summary": pd.DataFrame([
            {"generation_type": generation, "rows": stats.rows_by_generation[generation],
             "spans": stats.spans_by_generation[generation]}
            for generation in ("synthetic", "replacement")
        ]),
        "source_contribution": pd.DataFrame([
            {"entity_source": source, "rows": stats.source_rows[source], "span_attributions": stats.source_spans[source]}
            for source in sorted(stats.source_rows)
        ]),
        "quality_audit": pd.DataFrame([quality.__dict__]),
        "target_gaps": target_gaps,
        "skipped_labels": skipped_labels,
        "resource_audit": pd.DataFrame(RESOURCE_AUDIT),
        "rejected_examples": pd.DataFrame([
            {"label": label, "example": example}
            for label, examples in sorted(REJECTED_EXAMPLES.items()) for example in examples
        ]),
        "split_exceptions": pd.DataFrame(SPLIT_EXCEPTIONS),
    }
    with pd.ExcelWriter(XLSX_PATH, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name[:31], index=False)
            worksheet = writer.book[name[:31]]
            worksheet.freeze_panes = "A2"
            worksheet.auto_filter.ref = worksheet.dimensions
    return sheets


In [ ]:
def validate_final_csv() -> dict[str, Any]:
    split_counts = Counter()
    label_counts = Counter()
    value_splits: defaultdict[tuple[str, str], set[str]] = defaultdict(set)
    seen_ids: set[str] = set()
    seen_texts: set[str] = set()
    rows = 0
    with CSV_PATH.open(encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            rows += 1
            spans = json.loads(row["spans"])
            assert row["id"] not in seen_ids
            seen_ids.add(row["id"])
            normalized = Deduplicator.normalized(row["text"])
            assert normalized not in seen_texts
            seen_texts.add(normalized)
            assert row["is_saudi"] == "True"
            assert row["entity_source"].strip(), row["id"]
            assert not has_arabic(row["text"])
            assert row["final_split"] in {"train", "validation", "test"}
            split_counts[row["final_split"]] += 1
            for span in spans:
                assert row["text"][span["start"]:span["end"]] == span["value"]
                assert span["label"] in UNIVERSAL_LABELS
                label_counts[span["label"]] += 1
                if span["label"] == "PASSWORD":
                    decision = classify_password_span(row["text"], span["start"], span["end"])
                    assert decision.accept, (row["id"], decision.category, decision.reason)
                if span["label"] not in CLOSED_LABELS and span["label"] in REAL_POOL_LABELS:
                    value_splits[(span["label"], span["value"].casefold())].add(row["final_split"])
    leakage = [(label, value, sorted(splits)) for (label, value), splits in value_splits.items() if len(splits) > 1]
    assert not leakage, leakage[:5]
    assert label_counts["PASSWORD"] > 0, "PASSWORD target produced no disclosures"
    return {
        "rows": rows, "split_counts": dict(split_counts),
        "label_counts": dict(sorted(label_counts.items())),
        "password_spans": label_counts["PASSWORD"],
        "cross_split_entity_leakage": len(leakage),
    }

def file_fingerprint(path: Path) -> dict[str, Any]:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return {"file": path.name, "bytes": path.stat().st_size, "sha256": digest.hexdigest()}

def write_manifest(stats: Stats, quality: QualityAudit, effective_targets: dict[str, int],
                   final_validation: dict[str, Any]) -> dict[str, Any]:
    manifest = {
        "schema_version": 1,
        "generator_version": "saudi_mena.2",
        "ontology": {"num_entities": len(UNIVERSAL_LABELS), "num_bio_labels": 1 + 2 * len(UNIVERSAL_LABELS),
                     "entities": list(UNIVERSAL_LABELS)},
        "configuration": {
            "seed": SEED, "run_mode": RUN_MODE,
            "row_target_min": ROW_TARGET_MIN, "row_target_max": ROW_TARGET_MAX,
            "smoke_scale": SMOKE_SCALE, "effective_targets": dict(sorted(effective_targets.items())),
            "national_address_api_enabled": bool(NATIONAL_ADDRESS_API_KEY),
        },
        "artifacts": {
            "dataset": {**file_fingerprint(CSV_PATH), "rows": final_validation["rows"]},
            "summary_csv": file_fingerprint(SUMMARY_CSV_PATH),
            "summary_xlsx": file_fingerprint(XLSX_PATH),
        },
        "validation": final_validation,
        "quality_audit": dict(quality.__dict__),
        "rows_by_generation_type": dict(sorted(stats.rows_by_generation.items())),
        "resource_audit": RESOURCE_AUDIT,
        "reproducibility": {
            "notebook": "model/data/saudi_PII_Builder.ipynb",
            "output_directory": "model/data",
            "corpus_git_policy": "local and gitignored",
        },
    }
    MANIFEST_PATH.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return manifest


In [ ]:
# Code-level invariants for the standalone augmenter.
_split_revealing = re.compile(r"\b(?:benchmark|holdout|held-out|evaluation|training)\b", re.I)
assert not any(_split_revealing.search(value) for values in GENERAL_CONTEXT.values() for value in values)
assert set(DELETED_LABELS).isdisjoint(UNIVERSAL_LABELS)
assert len(UNIVERSAL_LABELS) == 31
assert "PASSWORD" in UNIVERSAL_LABELS
assert TARGETS["PASSWORD"] > 0
assert "WEIGHT" not in UNIVERSAL_LABELS
assert set(SKIPPED_FORMATS) == {"IDCARDNUM", "DRIVERLICENSENUM", "PASSPORTNUM"}
assert CSV_PATH.name == "saudi_mena_pii_dataset.csv"
assert SUMMARY_CSV_PATH.name == "saudi_mena_pii_summary.csv"
assert XLSX_PATH.name == "saudi_mena_pii_summary.xlsx"
assert MANIFEST_PATH.name == "saudi_mena_pii_manifest.json"


## 6. Build the dataset

The final cell creates the CSV, both summaries, validates every exported span, checks Latin-only text,
and verifies that open-vocabulary entity values do not cross split boundaries.


In [ ]:
STATS, QUALITY_AUDIT, EFFECTIVE_TARGETS, ENTITY_POOLS = build_dataset()
SUMMARY_SHEETS = write_summaries(STATS, QUALITY_AUDIT, ENTITY_POOLS)
FINAL_VALIDATION = validate_final_csv()
SAUDI_MANIFEST = write_manifest(STATS, QUALITY_AUDIT, EFFECTIVE_TARGETS, FINAL_VALIDATION)

print(f"Dataset: {CSV_PATH}")
print(f"Summary CSV: {SUMMARY_CSV_PATH}")
print(f"Summary workbook: {XLSX_PATH}")
print(f"Manifest: {MANIFEST_PATH}")
print(json.dumps(FINAL_VALIDATION, indent=2))
display(SUMMARY_SHEETS["split_summary"])
display(SUMMARY_SHEETS["quality_audit"])
display(SUMMARY_SHEETS["label_summary"][[
    "label", "total_spans", "unique_entity_count", "target_spans", "target_reached", "notes"
]])
if not SUMMARY_SHEETS["target_gaps"].empty:
    print("Targets not reached (quality and source reliability take priority):")
    display(SUMMARY_SHEETS["target_gaps"][["label", "total_spans", "target_spans", "notes"]])
